In [1]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.path.dirname("inpire.py"), '..'))
sys.path.append(parent_dir)

In [2]:
from inspire import InspireClassifier

import psutil
import time
import threading

from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import logging
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import BorderlineSMOTE

In [3]:
def monitor_resources(pid, usage_list, memory_list, stop_flag):
    process = psutil.Process(pid)
    while not stop_flag["stop"]:
        cpu = process.cpu_percent(interval=0.1)
        mem = process.memory_info().rss / (1024 ** 2)  # Convert to MB
        usage_list.append(cpu)
        memory_list.append(mem)

In [4]:
def measure_cpu_and_memory_usage(func):
    pid = psutil.Process().pid
    cpu_list = []
    mem_list = []
    stop_flag = {"stop": False}

    monitor_thread = threading.Thread(
        target=monitor_resources,
        args=(pid, cpu_list, mem_list, stop_flag)
    )
    monitor_thread.start()

    start_time = time.time()
    func()
    end_time = time.time()

    stop_flag["stop"] = True
    monitor_thread.join()

    avg_cpu = sum(cpu_list) / len(cpu_list) if cpu_list else 0
    peak_mem = max(mem_list) if mem_list else 0

    print("\n📊 Resource Usage:")
    print(f"  • Average CPU usage: {avg_cpu:.2f}%")
    print(f"  • Peak Memory usage: {peak_mem:.2f} MB")
    print(f"  • Execution time: {end_time - start_time:.2f} seconds")

# INSPIRE

In [5]:
# Polish companies bankruptcy
## Data
polish_companies_bankruptcy = fetch_ucirepo(id=365) 
  
X = polish_companies_bankruptcy.data.features
y = polish_companies_bankruptcy.data.targets
df = pd.DataFrame(X)
df["target"] = y
df.dropna(inplace=True)
y = df["target"].to_numpy()
X = df.drop("target", axis=1).to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)
minority_class = 1

## INSPIRE
### All features
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

In [6]:
measure_cpu_and_memory_usage(lambda: model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_per_step=1000,
    cleanup_=False,
    level=logging.CRITICAL,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7},
))


📊 Resource Usage:
  • Average CPU usage: 202.02%
  • Peak Memory usage: 520.58 MB
  • Execution time: 33.63 seconds


In [7]:
del model

# RFC + Borderline SMOTE

In [8]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

smote = BorderlineSMOTE(kind='borderline-1', random_state=42)

In [9]:
def fit_rfc(rfc, smote):
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    rfc.fit(X_resampled, y_resampled)

In [10]:
measure_cpu_and_memory_usage(lambda: fit_rfc(rfc=rfc, smote=smote))


📊 Resource Usage:
  • Average CPU usage: 100.70%
  • Peak Memory usage: 538.05 MB
  • Execution time: 77.70 seconds


In [11]:
del rfc
del smote